# jC_01 - Ablasi near-domain dengan 9 fitur (JURNAL, Tahap 1A)

**Pertanyaan:** kenaikan MCC dari far-domain (0,65) ke near-domain (0,85) apakah karena **domain lebih dekat**,
atau hanya karena near-domain punya **76 fitur** sedangkan far-domain cuma **9**?

**Cara menjawab:** jalankan FL near-domain (CIC2017 + CIC2018) hanya dengan **9 fitur yang sama** dengan far-domain
(durasi, jumlah/volume paket maju-mundur, rata-rata ukuran paket, laju). Semua lain identik dengan `jA_02`
(MLP, FedAvg, 100 ronde, 5 seed, protokol bebas-kebocoran). Satu-satunya yang berubah: jumlah fitur.

**Cara membaca hasil (disepakati sebelum melihat angka):**
- near@9 mendekati near@76 (~0,85)  -> keunggulan near-domain bukan karena jumlah fitur.
- near@9 mendekati far (~0,65)      -> jumlah fitur yang menjelaskan sebagian besar kenaikan.
- di antaranya                       -> keduanya berperan; laporkan apa adanya.

**Prasyarat:** `jA_01` sudah jalan (data near-domain 76 fitur di S3).
**Output:** `journal_out/neardomain9_fed_{seeds,agg}.csv` + S3 `federated_journal/neardomain/results/` dan
`run_J_NEAR9_s<seed>/eval_results.json`. Jalankan di SageMaker.

In [ ]:
import importlib, sys, subprocess
for m in ('numpy','boto3','scipy','pandas'):
    if importlib.util.find_spec(m) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',m],check=True)
import os, io, json, csv, time, numpy as np, pandas as pd, boto3

BUCKET='ssh-detection-features-232032302717'; REGION='ap-southeast-1'
BASE='federated_journal/neardomain'
CLIENTS=['cic2017','cic2018']
OUTDIR='journal_out'; os.makedirs(OUTDIR, exist_ok=True)
s3=boto3.client('s3',region_name=REGION)

# 9 fitur far-domain (CIC<->UNSW) dalam nama kolom near-domain (setelah alias)
NINE=['flow duration','total fwd packets','total backward packets',
      'total length of fwd packets','total length of bwd packets',
      'fwd packet length mean','bwd packet length mean',
      'flow bytes/s','bwd packets/s']

def s3_get_npz(key):
    obj=s3.get_object(Bucket=BUCKET,Key=key)
    return np.load(io.BytesIO(obj['Body'].read()),allow_pickle=True)
def s3_put_bytes(key,data): s3.put_object(Bucket=BUCKET,Key=key,Body=data)

TR={}; TE={}
for c in CLIENTS:
    dtr=s3_get_npz(f'{BASE}/data/{c}/{c}_train.npz'); dte=s3_get_npz(f'{BASE}/data/{c}/{c}_test.npz')
    feats=[str(f) for f in dtr['feats']]
    missing=[f for f in NINE if f not in feats]
    assert not missing, f'fitur tidak ada di near-domain: {missing}'
    idx=[feats.index(f) for f in NINE]
    TR[c]=(np.asarray(dtr['X'],float)[:,idx], np.asarray(dtr['y'],int))
    TE[c]=(np.asarray(dte['X'],float)[:,idx], np.asarray(dte['y'],int))
    print(c,'train',TR[c][0].shape,'pos=%.3f'%TR[c][1].mean(),'| test',TE[c][0].shape)
IN_DIM=len(NINE)
print('input dim =',IN_DIM,'| fitur:',NINE)

In [ ]:
HID=[128,64]
def init_weights(d, seed=42):
    rng=np.random.RandomState(seed); W=[]; L=[d]+HID+[1]
    for a,b in zip(L[:-1],L[1:]):
        W.append(rng.randn(a,b)*np.sqrt(2.0/a)); W.append(np.zeros(b))
    return W
def _relu(x): return np.maximum(0,x)
def _sig(x): return 1.0/(1.0+np.exp(-np.clip(x,-30,30)))
def forward(W,X):
    a1=X@W[0]+W[1]; z1=_relu(a1)
    a2=z1@W[2]+W[3]; z2=_relu(a2)
    a3=z2@W[4]+W[5]; out=_sig(a3).ravel()
    return out,(X,a1,z1,a2,z2,a3,out)
def train_local(W,X,y,epochs=1,lr=3e-3,batch=256,mu=0.0,W_global=None,seed=42):
    rng=np.random.RandomState(seed); n=len(y); W=[w.copy() for w in W]
    for _ in range(epochs):
        idx=rng.permutation(n)
        for s in range(0,n,batch):
            bi=idx[s:s+batch]; Xb=X[bi]; yb=y[bi].astype(float)
            out,(X_,a1,z1,a2,z2,a3,o)=forward(W,Xb)
            g=(o-yb)/len(yb)
            gW4=z2.T@g[:,None]; gb4=g.sum(0)
            d2=(g[:,None]@W[4].T)*(a2>0)
            gW2=z1.T@d2; gb2=d2.sum(0)
            d1=(d2@W[2].T)*(a1>0)
            gW0=X_.T@d1; gb0=d1.sum(0)
            for i,gv in enumerate([gW0,gb0,gW2,gb2,gW4,gb4]):
                gv=np.asarray(gv).reshape(W[i].shape)
                if mu>0 and W_global is not None: gv=gv+mu*(W[i]-W_global[i])
                W[i]=W[i]-lr*gv
    return W
def fedavg(wl,sizes):
    tot=float(sum(sizes)); out=[]
    for i in range(len(wl[0])):
        out.append(sum((sizes[k]/tot)*wl[k][i] for k in range(len(wl))))
    return out
def zscore_fit(X):
    mu=X.mean(0); sd=X.std(0); sd[sd==0]=1.0; return mu,sd
def zscore_apply(X,mu,sd): return (X-mu)/sd
def mcc(y,p):
    y=np.asarray(y).astype(int); p=np.asarray(p).astype(int)
    tp=int(((p==1)&(y==1)).sum()); tn=int(((p==0)&(y==0)).sum())
    fp=int(((p==1)&(y==0)).sum()); fn=int(((p==0)&(y==1)).sum())
    den=np.sqrt(float(tp+fp)*(tp+fn)*(tn+fp)*(tn+fn))
    return (tp*tn-fp*fn)/den if den>0 else 0.0
def recall(y,p):
    y=np.asarray(y).astype(int); p=np.asarray(p).astype(int)
    tp=int(((p==1)&(y==1)).sum()); fn=int(((p==0)&(y==1)).sum())
    return tp/(tp+fn) if (tp+fn)>0 else 0.0
def metrics(y,prob):
    p=(prob>=0.5).astype(int)
    return {'mcc':round(mcc(y,p),4),'recall':round(recall(y,p),4),
            'n':int(len(y)),'pos_rate':round(float(np.mean(y)),4)}
print('model + util siap (identik jA_02)')

In [ ]:
ROUNDS=100; LOCAL_EPOCHS=5; LR=3e-3

def run_one_seed(seed):
    zstat={c:zscore_fit(TR[c][0]) for c in CLIENTS}
    Xz={c:zscore_apply(TR[c][0],*zstat[c]) for c in CLIENTS}
    W=init_weights(IN_DIM, seed)
    for t in range(ROUNDS):
        cw=[]; cs=[]
        for c in CLIENTS:
            Wl=train_local(W,Xz[c],TR[c][1],epochs=LOCAL_EPOCHS,lr=LR,W_global=W,seed=seed)
            cw.append(Wl); cs.append(len(TR[c][1]))
        W=fedavg(cw,cs)
    W_fed=W
    def eval_model(Wm, norm='local'):
        if norm=='pooled':
            Xall=np.vstack([TR[c][0] for c in CLIENTS]); mup,sdp=zscore_fit(Xall)
        per={}; pa=[]; ya=[]
        for c in CLIENTS:
            Xte,yte=TE[c]
            mz,sz=(zstat[c] if norm=='local' else (mup,sdp))
            prob,_=forward(Wm, zscore_apply(Xte,mz,sz))
            per[c]=metrics(yte,prob); pa.append(prob); ya.append(yte)
        return {'per_client':per,'union':metrics(np.concatenate(ya),np.concatenate(pa))}
    res={'run':f'J_NEAR9_s{seed}','clients':CLIENTS,'in_dim':int(IN_DIM),'features':NINE,'seed':int(seed),
         'rounds':ROUNDS,'local_epochs':LOCAL_EPOCHS,'mode':'simulation(SageMaker)'}
    res['federated_local']=eval_model(W_fed,'local')
    res['federated_pooled_diagnostic']=eval_model(W_fed,'pooled')
    Xcat=np.vstack([TR[c][0] for c in CLIENTS]); ycat=np.concatenate([TR[c][1] for c in CLIENTS])
    mup,sdp=zscore_fit(Xcat)
    Wc=init_weights(IN_DIM,seed)
    Wc=train_local(Wc,zscore_apply(Xcat,mup,sdp),ycat,epochs=ROUNDS,lr=LR,seed=seed)
    res['centralized_pooled']=eval_model(Wc,'pooled')
    res['centralized_localnorm']=eval_model(Wc,'local')
    for c in CLIENTS:
        mz,sz=zstat[c]
        Wl=init_weights(IN_DIM,seed)
        Wl=train_local(Wl,zscore_apply(TR[c][0],mz,sz),TR[c][1],epochs=ROUNDS,lr=LR,seed=seed)
        res[f'local_only[{c}]']=eval_model(Wl,'local')
    return res
print('run_one_seed siap')

In [ ]:
SEEDS=[13,42,101,202,303]
all_res={}
for sd in SEEDS:
    t0=time.time(); r=run_one_seed(sd); dt=time.time()-t0
    all_res[sd]=r
    print(f"seed {sd}: fed(local)={r['federated_local']['union']['mcc']}  "
          f"cent(pooled)={r['centralized_pooled']['union']['mcc']}  "
          f"cent(local)={r['centralized_localnorm']['union']['mcc']}  "
          f"fed(pooled-diag)={r['federated_pooled_diagnostic']['union']['mcc']}  [{dt:.0f}s]")
    s3_put_bytes(f'{BASE}/run_J_NEAR9_s{sd}/eval_results.json', json.dumps(r,indent=2).encode('utf-8'))
print('selesai; eval per seed ter-upload ke S3')

In [ ]:
from scipy import stats
MODELS=['federated_local','centralized_localnorm','centralized_pooled',
        'federated_pooled_diagnostic','local_only[cic2017]','local_only[cic2018]']
def collect(model,scope,metric):
    vals=[]
    for sd in SEEDS:
        node=all_res[sd][model]
        m=(node['union'] if scope=='union' else node['per_client'][scope])
        vals.append(m[metric])
    return vals
def mean_ci95(vals):
    a=np.asarray(vals,float); n=len(a); mu=a.mean()
    if n<2: return mu,0.0
    return mu, a.std(ddof=1)/np.sqrt(n)*stats.t.ppf(0.975,n-1)

rows_seeds=[]; rows_agg=[]
for model in MODELS:
    for scope in ['union']+CLIENTS:
        for metric in ['mcc','recall']:
            vals=collect(model,scope,metric)
            for sd,v in zip(SEEDS,vals): rows_seeds.append(dict(model=model,scope=scope,metric=metric,seed=sd,value=v))
            mu,h=mean_ci95(vals)
            rows_agg.append(dict(model=model,scope=scope,metric=metric,mean=round(mu,4),ci95=round(h,4),
                                 values=';'.join(str(v) for v in vals)))
def write_csv(path,rows,fields):
    with open(path,'w',newline='') as f:
        w=csv.DictWriter(f,fieldnames=fields); w.writeheader(); w.writerows(rows)
write_csv(os.path.join(OUTDIR,'neardomain9_fed_seeds.csv'),rows_seeds,['model','scope','metric','seed','value'])
write_csv(os.path.join(OUTDIR,'neardomain9_fed_agg.csv'),rows_agg,['model','scope','metric','mean','ci95','values'])
for f in ('neardomain9_fed_seeds.csv','neardomain9_fed_agg.csv'):
    s3.upload_file(os.path.join(OUTDIR,f),BUCKET,f'{BASE}/results/{f}')
print('CSV tersimpan + upload ke S3 results/')

# perbandingan dengan near@76 (hasil jA_02 yang sudah ada di S3)
agg76=None
try:
    obj=s3.get_object(Bucket=BUCKET,Key=f'{BASE}/results/neardomain_fed_agg.csv')
    agg76=pd.read_csv(io.BytesIO(obj['Body'].read()))
except Exception as e:
    print('(hasil near@76 tak terbaca:',e,')')
FAR_FED=0.647   # far-domain federated union MCC (paper, EC2 5 seed)
print('\n=== UNION MCC: near@9 vs near@76 (mean +/- CI95) ===')
print(f"{'model':32s} {'near@9':>16s} {'near@76':>16s}")
for r in rows_agg:
    if r['scope']=='union' and r['metric']=='mcc':
        s9=f"{r['mean']:+.3f}+/-{r['ci95']:.3f}"
        s76='-'
        if agg76 is not None:
            q=agg76[(agg76.model==r['model'])&(agg76.scope=='union')&(agg76.metric=='mcc')]
            if len(q): s76=f"{q.iloc[0]['mean']:+.3f}+/-{q.iloc[0]['ci95']:.3f}"
        print(f"{r['model']:32s} {s9:>16s} {s76:>16s}")
fed9=[r for r in rows_agg if r['model']=='federated_local' and r['scope']=='union' and r['metric']=='mcc'][0]['mean']
print(f"\nfederated near@9 = {fed9:.3f} | near@76 = 0.851 | far (9 fitur, beda ekstraktor) = {FAR_FED}")
print('Baca hasil sesuai aturan di sel pertama (jangan menafsirkan sebelum angka keluar).')

## Catatan untuk naskah jurnal
- Ablasi ini mengisolasi jumlah fitur: ruang fitur near@9 identik (nama & definisi) dengan far-domain.
- Bila near@9 tetap jauh di atas far (0,647), kenaikan dapat dikaitkan ke jarak domain, bukan jumlah fitur.
- Bila tidak, tulis jujur bahwa jumlah fitur menjelaskan sebagian kenaikan.
- Hasil menggantikan kalimat "feature-controlled ablation = future work" di Section 5.1.